# Adaptive Gradient Boosting (AGB) Explained

Pega's **Adaptive Decision Manager (ADM)** uses **Adaptive Gradient Boosting
(AGB)** as its default predictive algorithm for Next-Best-Action recommendations.
AGB is an online-learning variant of gradient boosted trees that continuously
updates as new customer interactions arrive — no batch retraining required.

**Pega documentation:**
- [Adaptive Gradient boosting overview](https://docs.pega.com/bundle/alerts/page/platform/decision-management/adaptive-boosting-algorithm.html) — what AGB is and why it replaces Naïve Bayes
- [The Gradient boosting technique](https://docs.pega.com/bundle/alerts/page/platform/decision-management/adaptive-gradient-boosting.html) — deep dive: ensemble construction, gain formula, ADWIN pruning
- [Downloading a Gradient boosting Adaptive Model](https://docs.pega.com/bundle/alerts/page/platform/decision-management/gradient-boosting-model.html) — how to export the JSON from Prediction Studio
- [Interpreting Gradient boosting predictor importance](https://docs.pega.com/bundle/alerts/page/platform/decision-management/gradient-boosting-explanations.html) — reading the feature importance analysis in Prediction Studio

This notebook explains:

1. [**How a single decision tree works**](#1.-The-Building-Block:-A-Single-Decision-Tree) — nodes, splits, and the gain formula.
2. [**How trees combine**](#2.-The-Ensemble:-How-Trees-Build-on-Each-Other) into an ensemble, how the model grows over time, and
   the cold-start / warm-start behaviour.
3. [**How a customer is scored**](#3.-Scoring-a-Customer) — tracing the path through every tree.
4. [**Which predictors matter**](#4.-Feature-Importance:-Which-Predictors-Drive-the-Model?) — feature importance from split gains.
5. [**How to read model health**](#5.-Model-Health-at-a-Glance) — key metrics and what they signal.
6. [**Model AUC and calibration**](#6.-Model-AUC-and-Calibration) — how the model is evaluated and what the AUC score means.

All examples use the `pdstools` sample AGB model.


> **📦 Optional dependencies**
>
> This article uses the `pdstools` `adm` extra, and requires `pydot` and
> `graphviz` for tree visualizations. Install with your favorite package
> manager, e.g. `uv pip install "pdstools[adm]" pydot graphviz`.


In [ ]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook separately

import plotly.io as pio

pio.renderers.default = "notebook_connected"


In [ ]:
import json
import polars as pl
from math import exp
from great_tables import GT
from pdstools import datasets

AGBModel = datasets.sample_trees()
# To use your own model, export the JSON from Prediction Studio and load it with:
# from pdstools.adm.trees import ADMTreesModel
# AGBModel = ADMTreesModel.from_file("path/to/model_export.json")
# See: https://docs.pega.com/bundle/platform/page/platform/pega-ai-tools/export-model-data-prediction-studio.html
print(f"Loaded model: {len(AGBModel.model)} trees, Pooled AUC={AGBModel.metrics['auc']:.4f}")
print(f"Training data: {AGBModel.metrics['response_positive_count']:,} positives, "
      f"{AGBModel.metrics['response_negative_count']:,} negatives")


**Caveat:** Pooled AUC—computed by mixing prediction scores and outcomes across all actions before drawing a single ROC curve—is systematically inflated by action base-rate differences and action-mix. It is therefore not a reliable measure of the discriminative power of any individual action model. For model-performance assessment, prefer the response-count weighted average of per-action AUC values, explained in **Pooled vs weighted-average AUC** below.

### Pooled vs weighted-average AUC

**Pooled AUC** mixes prediction scores and outcomes from multiple actions into one combined ROC calculation. As the whitepaper shows, this number is structurally inflated by cross-action base-rate separation, so a high pooled AUC can coexist with weak per-action models.

**Weighted-average AUC** is the response-count weighted average of per-action AUC values. This is the honest, actionable portfolio metric because it answers the operational question that matters in NBA: does each action model rank likely responders above likely non-responders within its own action?

In practice, report weighted-average AUC as the primary portfolio measure and treat pooled AUC only as a descriptive aggregate, not as evidence of model quality.